In [39]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [40]:
eval_iters = 200
batch_size = 64
eval_interval = 500
block_size = 256
max_iters = 5000
learning_rate = 3e-4
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print ("device: ", device)

device:  cuda


In [42]:
with open ("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [43]:
print("length of dataset in chars: ", len(text))

length of dataset in chars:  1115394


In [44]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [45]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("characters:", "".join(chars))
print("vocabulary size: ", vocab_size)

characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocabulary size:  65


In [46]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join (itos[i] for i in l)

In [47]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [48]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:100])

torch.Size([1115394]) <built-in method type of Tensor object at 0x7ea951856300>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [49]:
n = int(0.9 * len (data))

train_data = data[:n]
val_data = data[n:]

In [50]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb =  get_batch("train")

print ("inputs")
print(xb.shape)
print(xb)

print ("targets")
print(yb.shape)
print(yb)


inputs
torch.Size([64, 256])
tensor([[ 0, 26, 53,  ..., 56, 43, 47],
        [60, 43, 56,  ..., 56,  1, 41],
        [26, 21, 33,  ..., 26, 21, 13],
        ...,
        [ 5, 57,  1,  ...,  1, 35, 47],
        [56, 53, 53,  ..., 59, 50, 42],
        [42, 47, 56,  ..., 39, 56,  1]], device='cuda:0')
targets
torch.Size([64, 256])
tensor([[26, 53, 58,  ..., 43, 47, 45],
        [43, 56,  1,  ...,  1, 41, 53],
        [21, 33, 31,  ..., 21, 13, 10],
        ...,
        [57,  1, 52,  ..., 35, 47, 50],
        [53, 53, 58,  ..., 50, 42,  1],
        [47, 56, 43,  ..., 56,  1, 51]], device='cuda:0')


In [51]:
class Head (nn.Module):

    def __init__ (self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**(-0.5)
        wei = wei.masked_fill (self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax (wei, dim = -1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

In [52]:
class MultiHeadAttention(nn.Module):

    def __init__ (self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList ([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear (n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out

In [53]:
class FeedForward(nn.Module):

    def __init__ (self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
            )

    def forward(self, x):
        return self.net(x)

In [54]:
class Block(nn.Module):

    def __init__ (self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward (self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [55]:
class BigramLanguageModel(nn.Module):

    def __init__ (self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential (* [Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward (self, idx, targets = None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange (T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate (self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits [:, -1, :]
            probs = F.softmax (logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

model = BigramLanguageModel()
model = model.to(device)

In [56]:
total_params = sum(p.numel() for p in model.parameters())
print("model parameters count: ", total_params)

model parameters count:  10788929


In [57]:
logits, loss = model(xb, yb)

print(logits.shape)
print(loss)

torch.Size([16384, 65])
tensor(4.3387, device='cuda:0', grad_fn=<NllLossBackward0>)


In [58]:
idx = torch.zeros((1, 1), dtype=torch.long, device = device)
print (decode(model.generate(idx, max_new_tokens=100)[0].tolist()))


DfmMX$l?bM?Fwc
abS!SNpzBe!NLsPOhppDl!KCWyAIEI-zBg?Zqbld$kzFH dTJgnCcaAB zdwCmsxcUejMThsmtHKRe:tcB--g


In [59]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [60]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range (eval_iters):
            X, Y = get_batch(split)
            _, loss = model (X, Y)
            losses [k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [61]:
for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses["train"]:.4f}, val loss = {losses["val"]:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.3360, val loss = 4.3324
step 500: train loss 2.0151, val loss = 2.0982
step 1000: train loss 1.6088, val loss = 1.7848
step 1500: train loss 1.4452, val loss = 1.6391
step 2000: train loss 1.3460, val loss = 1.5677
step 2500: train loss 1.2828, val loss = 1.5258
step 3000: train loss 1.2303, val loss = 1.5016
step 3500: train loss 1.1872, val loss = 1.4870
step 4000: train loss 1.1491, val loss = 1.4744
step 4500: train loss 1.1118, val loss = 1.4694


In [62]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print (decode(model.generate(context, max_new_tokens=500)[0].tolist()))


But, I'll enterich be, be our souls on the
more or bittering with their weapous
and yourself: but they to stab'd to some kiss
dealy to that you finges in the followers
And, and beir morning, the lord's made appliance
Of buring watchman advices: you have named me
To bear us served advice and sickly
The fellish power beseem to see, the realm.

BRUTUS:
Good toid her! Why the prince in her;
These both borned slacks monour sway upon
As cold not the partness are of all.

CORIOLANUS:
Play thee, good gi
